# 01 — Start Here: Training + Export Fiorell.IA

Questo notebook serve per addestrare l’adapter LoRA di Fiorell.IA in Google Colab e salvare lo ZIP finale su Google Drive.

Devi solo eseguire le celle dall’alto verso il basso. Non modificare codice, salvo i parametri nella sezione 0.

## Cosa produce

- `fiorellia_lora_adapter.zip`
- `training_summary.json`
- `training_final_summary.md`


## 0 — Configurazione semplice

Controlla solo queste righe. In Colab la configurazione standard dovrebbe funzionare senza modifiche.

In [ ]:
from pathlib import Path
import sys, json, subprocess

RUN_ENV = 'colab'  # colab oppure local
REPO_URL = 'https://github.com/TheGenesisAIStory/regulatory-insight-engine.git'
REPO_ROOT = Path('/content/regulatory-insight-engine') if RUN_ENV == 'colab' else Path('/Users/itsgennymac/Documents/GitHub/regulatory-insight-engine')
DRIVE_ROOT = Path('/content/drive/MyDrive/fiorellia/training_final') if RUN_ENV == 'colab' else REPO_ROOT / 'artifacts/fiorellia/training_final'
CONFIG_PATH = REPO_ROOT / 'fiorellia/training/configs/config_lora_behavior_20260421.yaml'
TRAIN_SCRIPT_CANDIDATES = [
    REPO_ROOT / 'fiorellia/training/train_lora_behavior_v1.py',
    REPO_ROOT / 'fiorellia/training/train_lora_behavior.py',
]
ADAPTER_ZIP = DRIVE_ROOT / 'fiorellia_lora_adapter.zip'
TRAINING_SUMMARY_JSON = DRIVE_ROOT / 'training_summary.json'
TRAINING_SUMMARY_MD = DRIVE_ROOT / 'training_final_summary.md'

print('Repository:', REPO_ROOT)
print('Output Drive/local:', DRIVE_ROOT)
print('Config:', CONFIG_PATH)

## 1 — Preparazione Colab

Questa sezione monta Google Drive e clona il repository se non è già presente.

Se Colab chiede autorizzazione a Google Drive, accetta.

In [ ]:
if RUN_ENV == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    if not REPO_ROOT.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(REPO_ROOT))
print('Ambiente pronto.')

## 2 — Installazione dipendenze

Esegui questa cella una sola volta. Se Colab chiede di riavviare il runtime, riavvia e poi riparti dalla sezione 0.

In [ ]:
%pip install -q \
  "transformers>=4.45,<4.52" \
  "datasets>=2.20,<3.0" \
  "accelerate>=0.33,<1.0" \
  "peft>=0.12,<0.16" \
  "trl>=0.9,<0.13" \
  "bitsandbytes>=0.43,<0.46" \
  "safetensors>=0.4" \
  "pyyaml>=6.0"
print('Dipendenze installate.')

## 3 — Controlli preliminari

Questa cella controlla GPU, config e dataset. Se qualcosa non va, mostra un errore leggibile.

In [ ]:
from fiorellia.training.fiorellia_colab_pipeline import (
    check_cuda, load_config, validate_config, require_file, write_json
)

gpu = check_cuda(require_gpu=True)
config = load_config(CONFIG_PATH)
resolved = validate_config(config, REPO_ROOT)
train_script = next((p for p in TRAIN_SCRIPT_CANDIDATES if p.exists()), None)
if train_script is None:
    raise FileNotFoundError('Script training non trovato. Controlla fiorellia/training/.')

preflight = {
    'gpu': gpu,
    'base_model_name': config.get('base_model_name'),
    'dataset_path': str(resolved['dataset_path']),
    'output_dir': str(resolved['output_dir']),
    'train_script': str(train_script),
}
write_json(preflight, DRIVE_ROOT / 'preflight_training.json')
print(json.dumps(preflight, indent=2, ensure_ascii=False))
print('Preflight completato: puoi avviare il training.')

## 4 — Training LoRA

Questa cella avvia il training. Può richiedere tempo. Attendi il messaggio finale senza chiudere Colab.

In [ ]:
cmd = [sys.executable, str(train_script), '--config', str(CONFIG_PATH)]
print('+', ' '.join(cmd))
subprocess.run(cmd, cwd=str(REPO_ROOT), check=True)
print('Training completato.')

## 5 — Export adapter ZIP

Questa cella controlla che l’adapter esista e crea lo ZIP finale su Drive.

In [ ]:
from fiorellia.training.fiorellia_colab_pipeline import zip_adapter, validate_adapter_zip

adapter_dir = resolved['output_dir']
created_zip = zip_adapter(adapter_dir, ADAPTER_ZIP)
validate_adapter_zip(created_zip)
print('Adapter ZIP creato e validato:', created_zip)

## 6 — Conclusione finale training

Questa cella salva un riepilogo finale semplice. Se vedi `TRAINING OK`, puoi passare al notebook di eval.

In [ ]:
from datetime import datetime
from fiorellia.training.fiorellia_colab_pipeline import write_json

summary = {
    'status': 'TRAINING OK',
    'timestamp': datetime.utcnow().isoformat() + 'Z',
    'adapter_zip': str(ADAPTER_ZIP),
    'config': str(CONFIG_PATH),
    'base_model_name': config.get('base_model_name'),
    'next_step': 'Apri fiorellia/eval/02_start_here_eval_decision.ipynb',
}
write_json(summary, TRAINING_SUMMARY_JSON)
TRAINING_SUMMARY_MD.write_text(
    '# Conclusione finale training Fiorell.IA\n\n'
    'Esito: **TRAINING OK**\n\n'
    f'Adapter ZIP: `{ADAPTER_ZIP}`\n\n'
    'Prossimo passo: aprire il notebook di eval e usare questo ZIP.\n',
    encoding='utf-8'
)
print(json.dumps(summary, indent=2, ensure_ascii=False))
print('CONCLUSIONE FINALE: TRAINING OK')